<a href="https://colab.research.google.com/github/allenphos/ML-course-ua-/blob/main/Copy_of_HW_%D0%9C%D0%B5%D1%82%D0%BE%D0%B4%D0%B8_%D0%BF%D0%BE%D0%BD%D0%B8%D0%B6%D0%B5%D0%BD%D0%BD%D1%8F_%D1%80%D0%BE%D0%B7%D0%BC%D1%96%D1%80%D0%BD%D0%BE%D1%81%D1%82%D1%96.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Домашнє завдання: Пониження розмірностей для Аналізу Портретів Клієнтів

#### Контекст
В цьому ДЗ ми попрацюємо з методами пониження розмірності на наборі даних для задачі аналізу портретів клієнтів (Customer Personality Analysis). **В попередньому ДЗ ми працювали з цими даними використовуючи кластеризацію, зараз використаємо кластеризацію і візуалізауємо результати з різними методами.**

Customer Personality Analysis - це аналіз різних сегментів клієнтів компанії. Цей аналіз дозволяє бізнесу краще розуміти своїх клієнтів і полегшує процес адаптації продуктів під конкретні потреби, поведінку та інтереси різних типів клієнтів.

Аналіз портретів клієнтів допомагає бізнесу змінювати свій продукт на основі цільової аудиторії, розділеної на різні сегменти. Наприклад, замість того, щоб витрачати гроші на маркетинг нового продукту для всіх клієнтів у базі даних компанії, бізнес може проаналізувати, який сегмент клієнтів найімовірніше придбає продукт, і потім зосередити маркетингові зусилля лише на цьому сегменті.

#### Вхідні дані
Вам надано набір даних з такими атрибутами:

**Характеристики користувачів:**
- `ID`: Унікальний ідентифікатор клієнта
- `Year_Birth`: Рік народження клієнта
- `Education`: Рівень освіти клієнта
- `Marital_Status`: Сімейний стан клієнта
- `Income`: Річний дохід домогосподарства клієнта
- `Kidhome`: Кількість дітей у домогосподарстві клієнта
- `Teenhome`: Кількість підлітків у домогосподарстві клієнта
- `Dt_Customer`: Дата реєстрації клієнта у компанії
- `Recency`: Кількість днів з моменту останньої покупки клієнта
- `Complain`: 1, якщо клієнт скаржився за останні 2 роки, 0 - якщо ні

**Продукти:**
- `MntWines`: Сума, витрачена на вино за останні 2 роки
- `MntFruits`: Сума, витрачена на фрукти за останні 2 роки
- `MntMeatProducts`: Сума, витрачена на м'ясні продукти за останні 2 роки
- `MntFishProducts`: Сума, витрачена на рибні продукти за останні 2 роки
- `MntSweetProducts`: Сума, витрачена на солодощі за останні 2 роки
- `MntGoldProds`: Сума, витрачена на золото за останні 2 роки

**Акції:**
- `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
- `AcceptedCmp1`: 1, якщо клієнт прийняв пропозицію у першій кампанії, 0 - якщо ні
- `AcceptedCmp2`: 1, якщо клієнт прийняв пропозицію у другій кампанії, 0 - якщо ні
- `AcceptedCmp3`: 1, якщо клієнт прийняв пропозицію у третій кампанії, 0 - якщо ні
- `AcceptedCmp4`: 1, якщо клієнт прийняв пропозицію у четвертій кампанії, 0 - якщо ні
- `AcceptedCmp5`: 1, якщо клієнт прийняв пропозицію у п'ятій кампанії, 0 - якщо ні
- `Response`: 1, якщо клієнт прийняв пропозицію в останній кампанії, 0 - якщо ні

**Взаємодія з компанією:**
- `NumWebPurchases`: Кількість покупок, зроблених через вебсайт компанії
- `NumCatalogPurchases`: Кількість покупок, зроблених за каталогом
- `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
- `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Для початку, запустіть код нижче. Всі ці кроки ми робили в попередньому ДЗ і для того, щоб результати кластеризації у нас були схожими, потрібно аби передобробка була однаковою.

In [3]:
import pandas as pd

# 1. Завантаження даних
df = pd.read_csv('drive/MyDrive/Colab Notebooks/data/marketing_campaign.csv', sep='\t')

# 2. Обробка пропущених значень
df['Income_not_filled'] = df.Income.isna()
df.Income = df.Income.fillna(-1)

# 3. Обробка дати реєстрації
df.Dt_Customer = pd.to_datetime(df.Dt_Customer, format='%d-%m-%Y')
today = df.Dt_Customer.max()
df['days_lifetime'] = (today - df.Dt_Customer).dt.days
df['years_customer'] = df.Year_Birth.apply(lambda x: today.year - x)

# 4. Категоризація рівня освіти
df_education = pd.get_dummies(df.Education, prefix='education').astype(int)
df = pd.concat([df, df_education], axis=1)

# 5. Очищення сімейного стану
marital_status_map = {'Alone': 'Single', 'Absurd': 'Else', 'YOLO': 'Else'}
df['Marital_Status_clean'] = df.Marital_Status.map(marital_status_map)
df_ms = pd.get_dummies(df.Marital_Status_clean, prefix='marital').astype(int)
df = pd.concat([df, df_ms], axis=1)

# 6. Форматування доходу і видалення викиду
df.Income = df.Income.astype(int)
df = df[df.Income != 666666]

# 7. Створення фінального набору даних
X = df.drop(['ID', 'Dt_Customer', 'Education', 'Marital_Status', 'Marital_Status_clean'], axis=1)
X.reset_index(drop=True, inplace=True)

### Завдання 1: Виконання кластеризації та пониження розмірності для візуалізації результатів

Ваше завдання — провести кластеризацію клієнтів та візуалізувати результати кластеризації, використовуючи метод головних компонент (PCA) для пониження розмірності даних.

#### Інструкції:

1. **Вибір ключових характеристик:**
   Давайте обмежимось тільки наступними хараткеристиками для кластеризації цього разу:
   - `Income`: Річний дохід домогосподарства клієнта
   - `Recency`: Кількість днів з моменту останньої покупки клієнта
   - `NumStorePurchases`: Кількість покупок, зроблених безпосередньо у магазинах
   - `NumDealsPurchases`: Кількість покупок, зроблених з використанням знижок
   - `days_lifetime`: Кількість днів з моменту реєстрації клієнта у компанії
   - `years_customer`: Вік клієнта
   - `NumWebVisitsMonth`: Кількість відвідувань вебсайту компанії за останній місяць
   Відберіть в наборі даних `X` лише ці характеристики.

2. **Нормалізація даних:**
   Використайте метод `MinMaxScaler` для нормалізації значень обраних характеристик.

3. **Кластеризація:**
   Проведіть кластеризацію клієнтів, використовуючи метод `KMeans` з трьома кластерами.

4. **Пониження розмірності:**
   Використайте метод головних компонент (PCA) для пониження розмірності даних до трьох компонент.

5. **Візуалізація результатів:**
   Використовуючи plolty express побудуйте 3D-графік розподілу клієнтів у просторі трьох головних компонент, де кольором позначено кластери.

6. **Опишіть, що спостерігаєте:**
   Чи кластеризація чітко розділила дані?

Далі ми детальніше проінтерпретуємо результати візуалізації і пониження розмірностей.

In [6]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

X = X[['Income', 'Recency', 'NumStorePurchases', 'NumDealsPurchases', 'days_lifetime', 'years_customer', 'NumWebVisitsMonth']]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

In [7]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
pca.fit(X)

PCA(n_components=3)

In [8]:
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA # Import PCA


pca = PCA(n_components=3, random_state=0)
pca_digits = pca.fit_transform(X_scaled)  # Use X_scaled instead of digits.data

# Create a DataFrame for Plotly Express
df = pd.DataFrame(data=pca_digits, columns=['PC1', 'PC2', 'PC3'])
df['target'] = cluster_labels  # Use cluster_labels instead of digits.target

# Create an interactive 3D scatter plot
fig = px.scatter_3d(
    df,
    x='PC1',
    y='PC2',
    z='PC3',
    hover_data='target',
    color='target',
    title='3D Scatter Plot of Customer Data with PCA'
)

# Show the figure
fig.show()

Кластеризація чітко розділила дані на 3 частини.

### Завдання 2: Аналіз результатів пониження розмірності

1. **Розрахунок частки поясненої дисперсії:**
   Визначте, яка частка загальної варіації даних пояснюється кожною з трьох головних компонент (PC1, PC2, PC3) за допомогою атрибуту `explained_variance_ratio_` об'єкта PCA. Виведіть результат на екран.

2. **Розрахунок кумулятивної частки поясненої дисперсії:**
   Обчисліть кумулятивну частку поясненої дисперсії для трьох головних компонент, щоб зрозуміти, скільки варіації даних пояснюється першими кількома компонентами.

In [9]:
pca.explained_variance_ratio_

array([0.30203449, 0.2866534 , 0.25122414])

In [13]:
pca = PCA(n_components=3)
pca_data = pca.fit_transform(X_scaled)

In [14]:
pca.explained_variance_ratio_.cumsum()

array([0.30203449, 0.5886879 , 0.83991203])

PC1 alone explains 30% of the variance.

PC1 and PC2 together explain 59% of the variance.

PC1, PC2, and PC3 together explain 84% of the variance.

### Завдання 3: Інтерпретація "Loadings"

Продовжуємо інтерпретацію результатів `PCA`і познайомимось з новим поняттям `loadings`, яке допоможе нам знайти звʼязок між головними компонентами і оригінальними ознаками в наборі даних.

Ми зараз побудували візуалізацію кластерів точок даних в просторі трьох головних компонент. Але хочеться знайти звʼязок між головними компонентами і оригінальними ознаками. Для розуміння, які початкові характеристики даних мають найбільший вплив на ці головні компоненти, ми можемо використати атрибут `components_` методу `PCA`.

#### Що таке `pca.components_`?

`pca.components_` — це масив, який містить коефіцієнти (або "ваги"), що показують внесок кожної вихідної ознаки у кожну з головних компонент. Ці коефіцієнти ще називаються **"loading"** або "навантаженнями" компонент.

- **Loadings** (`навантаження`) відображають важливість кожної змінної (ознаки) для відповідної головної компоненти. Вони показують, яким чином змінні поєднуються, щоб утворити нові, зменшені вимірювання.
- Якщо коефіцієнт має високе абсолютне значення (як позитивне, так і негативне), це вказує на те, що відповідна змінна сильно впливає на головну компоненту.

#### Саме завдання
Ваше завдання — обчислити "навантаження" для кожної з головних компонент і інтерпретувати результати.

1. **Обчислення loadings для компонент:**
   Використайте атрибут `components_` об'єкта PCA для створення DataFrame, який відображатиме внесок кожної вихідної ознаки в кожну головну компоненту.

2. **Інтерпретація результатів:**
   Виведіть значення "навантажень" і проаналізуйте, які ознаки найбільше впливають на кожну головну компоненту.

In [17]:
# Get the feature names from the original DataFrame
feature_names = X.columns

# Create a DataFrame with the loadings
loadings_df = pd.DataFrame(
    data=pca.components_.T,  # Transpose to have features as rows
    columns=['PC1', 'PC2', 'PC3'],
    index=feature_names
)

# Display the loadings DataFrame
print(loadings_df)

                        PC1       PC2       PC3
Income             0.063557 -0.047082  0.373826
Recency            0.475786  0.878876  0.029910
NumStorePurchases  0.284282 -0.187195  0.816668
NumDealsPurchases  0.103049 -0.059324 -0.050687
days_lifetime      0.821912 -0.431061 -0.305490
years_customer     0.012658  0.001022  0.080411
NumWebVisitsMonth  0.049530 -0.031377 -0.300089


На PC1 впливають найбільше ознаки days_lifetime            
На PC2 впливають найбільше ознаки Recency   
На PC3 впливають найбільше ознаки NumStorePurchases    

###Завдання 4
Давайте проаналізуємо "навантаження" (**loadings**) для трьох головних компонент після вилучення ознаки `Income`. Це допоможе нам зрозуміти, як змінилася важливість інших ознак для кожної головної компоненти, коли одна з ключових характеристик (`Income`) була вилучена.

#### Кроки для проведення аналізу і ваше завдання:

1. Видаліть ознаку `Income` з нашого набору даних `X` і повторно виконайте PCA (метод головних компонент) для отримання нових "навантажень".

2. Обчисліть нові "навантаження" для трьох головних компонент на наборі даних без `Income`

3. Проаналізуйте, які ознаки мають найбільший вплив на кожну головну компоненту після вилучення `Income`.

4. Перегляньте, наскільки кожна з головних компонент пояснює дисперсію в даних без ознаки `Income`.

In [18]:
X_no_income = X.drop('Income', axis=1)

# Re-scale the data without 'Income'
scaler = MinMaxScaler()
X_scaled_no_income = scaler.fit_transform(X_no_income)

# Re-run PCA with 3 components
pca_no_income = PCA(n_components=3)
pca_no_income.fit(X_scaled_no_income)

PCA(n_components=3)

In [20]:
# Get the feature names from the original DataFrame
feature_names_no_income = X_no_income.columns

# Create a DataFrame with the new loadings
loadings_df_no_income = pd.DataFrame(
    data=pca_no_income.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=feature_names_no_income
)

# Display the new loadings DataFrame
print(loadings_df_no_income)

                        PC1       PC2       PC3
Recency            0.500712  0.865392  0.008648
NumStorePurchases  0.226953 -0.145537  0.924549
NumDealsPurchases  0.105012 -0.065833 -0.027582
days_lifetime      0.825793 -0.472036 -0.232275
years_customer     0.007110  0.005913  0.079808
NumWebVisitsMonth  0.069051 -0.052233 -0.289919


Нічого не змінилось після видаленя колонки про прибуток.

In [22]:
# Get the explained variance ratio for each component
explained_variance_ratio_no_income = pca_no_income.explained_variance_ratio_

# Print the explained variance ratio
print("Explained Variance Ratio:", explained_variance_ratio_no_income)

# Calculate and print the cumulative explained variance
cumulative_variance_no_income = explained_variance_ratio_no_income.cumsum()
print("Cumulative Explained Variance:", cumulative_variance_no_income)

Explained Variance Ratio: [0.32149032 0.30545488 0.23446936]
Cumulative Explained Variance: [0.32149032 0.6269452  0.86141456]


PC1 explains 32% of the variance.

PC2 explains 30% of the variance.

PC3 explains 23% of the variance.

PC1 alone explains 32% of the variance.

PC1 and PC2 together explain 63% of the variance.

PC1, PC2, and PC3 together explain 86% of the variance.

### Завдання 5: Візуалізація кластеризації за допомогою t-SNE

Ваше завдання — використати метод t-SNE для візуалізації результатів кластеризації клієнтів у двовимірному просторі. Метод t-SNE допомагає знизити розмірність даних та зберегти локальні структури в даних, що робить його ефективним для візуалізації високорозмірних даних. Ми також зможемо порівняти результат цього методу з РСА.

1. Використайте метод t-SNE для зниження розмірності до 2х вимірів даних, які включають ознаки всі, що і в завданні 1, а також були відмасштабовані перед пониженням розмірностей.

2. Створіть новий DataFrame з координатами, отриманими після застосування t-SNE, та додайте до нього мітки кластерів.

3. Побудуйте інтерактивний 2D-графік розподілу клієнтів, де кольором буде позначено різні кластери і проаналізуйте графік з рекомендаціями нижче (можливо треба буде вивести додаткові візуалізації чи таблиці для інтерпретації, але треба прям зрозуміти, які ознаки формують який кластер і чим кластери відрізняються одне від одного).

  **Опишіть отримані кластери з точки зору ознак.**

4. Опишіть відмінність графіка tSNE від PCA.

#### ЯК можна інтерпретувати з t-SNE?

Хоча t-SNE не надає "компонентів" як РСА, він забезпечує низьковимірне представлення даних, яке можна візуально інтерпретувати:

- **Кластери:** t-SNE особливо добре показує кластери подібних точок. Якщо ви бачите чітко визначені кластери на графіку t-SNE, це свідчить про наявність груп схожих спостережень у ваших даних. Проаналізувати їх можемо, якщо додамо дані в `hover_data` або якщо якісь з даних виведемо як розмір чи форма точок на візуалізації. Також корисно може бути вивести середні значення ознак по кластерам.
- **Локальна структура:** Відносне розташування точок одного кластеру на графіку t-SNE може допомогти вам зрозуміти, які дані подібні між собою.
- **Глобальна структура:** Будьте обережні; t-SNE менш надійний для відображення глобальних структур (наприклад, відстаней між кластерами) у порівнянні з PCA, бо t-SNE націлений на збереження саме локальних структур.

In [25]:
from sklearn.manifold import TSNE

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=0)
tsne_data = tsne.fit_transform(X_scaled)

# Create a DataFrame with t-SNE coordinates and cluster labels
tsne_df = pd.DataFrame(data=tsne_data, columns=['t-SNE 1', 't-SNE 2'])
tsne_df['Cluster'] = cluster_labels

# Add the original data for hover_data
tsne_df = pd.concat([tsne_df, X], axis=1)  # Add original features

# Create an interactive 2D scatter plot
fig = px.scatter(
    tsne_df,
    x='t-SNE 1',
    y='t-SNE 2',
    color='Cluster',
    hover_data=X.columns, # Add original feature values to hover data
    title='t-SNE Visualization of Customer Clusters'
)
fig.show()

# Analyze clusters based on feature characteristics
cluster_means = X.groupby(cluster_labels).mean()
print(cluster_means)

         Income    Recency  NumStorePurchases  NumDealsPurchases  \
0  44990.324561  21.352130           4.264411           2.327068   
1  69681.154622  51.163025          10.275630           2.273950   
2  44622.521277  73.878251           4.078014           2.356974   

   days_lifetime  years_customer  NumWebVisitsMonth  
0     337.989975       43.987469           5.849624  
1     398.705882       46.610084           3.880672  
2     336.508274       45.346336           5.822695  


In [ ]:
Cluster 0:

Income: 44990.32
Recency: 21.35
NumStorePurchases: 4.26
NumDealsPurchases: 2.33
days_lifetime: 337.99
years_customer: 43.99
NumWebVisitsMonth: 5.85

Cluster 1:

Income: 69681.15
Recency: 51.16
NumStorePurchases: 10.28
NumDealsPurchases: 2.27
days_lifetime: 398.71
years_customer: 46.61
NumWebVisitsMonth: 3.88

Cluster 2:

Income: 44622.52
Recency: 73.88
NumStorePurchases: 4.08
NumDealsPurchases: 2.36
days_lifetime: 336.51
years_customer: 45.35
NumWebVisitsMonth: 5.82

Перший і третій майже однакові, окрім Recency (кількість днів з моменту останньої покупки клієнта), що вказує на погану кластеризацію.

З t-SNE можно чітко побачити кластери, він добре візуалізує складні структури. Але виконувався код повільніше ніж PCA. t-SNE краще підходить для дослідження стосунків і взаємозв'язків між сусідніми точками даних.